# Fake News Detection — Full Pipeline

**Project:** A Comparative Study of Classical and Deep Learning Approaches for Fake News Detection

**Dataset:** William Lifferth's *Fake News* dataset from Kaggle (Kaggle Competition `fake-news`). This dataset is more challenging than the popular Bisaillon Real-vs-Fake dataset because the real news comes from a wider variety of mainstream sources (not just Reuters), reducing the publisher-style leakage that makes other fake-news datasets trivially easy.

**Pipeline stages:**
1. **Stage 1** — Data loading, EDA, text preprocessing, TF-IDF vectorization
2. **Stage 2** — Classical models: Naive Bayes, Logistic Regression, Linear SVM, XGBoost
3. **Stage 3** — Bidirectional LSTM (deep learning baseline)
4. **Stage 4** — Final comparison, McNemar's test, error analysis, export

**Expected accuracy range:** 92–96% — meaningfully harder than the trivial Bisaillon dataset (which yields ~99% accuracy because real news comes only from Reuters and is stylistically distinguishable in obvious ways).

## 1. Imports and Setup

In [ ]:
import os
import re
import string
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from scipy.sparse import save_npz, load_npz

# Set plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Download NLTK resources
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Ensure output directory exists
os.makedirs('/kaggle/working/', exist_ok=True)

print('Setup complete.')

## 2. Loading the Data

The William Lifferth fake-news dataset (originally from a Kaggle competition) is mirrored on GitHub. We download it directly from a public raw GitHub URL — **no Kaggle account, no API key, no rules acceptance required**. The download is a single file, `train.csv`, ~94 MB.

**Schema:** `id`, `title`, `author`, `text`, `label` (1 = unreliable/fake, 0 = reliable/real)

**Note on the label convention.** This dataset uses `1 = fake, 0 = real`. To keep our analysis consistent with how we'd typically frame the problem (positive class = "real news"), **we flip the labels** so that `1 = real, 0 = fake` throughout. Note that this flip happens later, in Section 3.

In [ ]:
# Download train.csv from a public GitHub mirror (no auth needed)
import os
import urllib.request

DATA_DIR = '/kaggle/working/dataset/'
TRAIN_PATH = os.path.join(DATA_DIR, 'train.csv')
TRAIN_URL = ('https://raw.githubusercontent.com/'
             'AnitaKatuwal04/Fake_news_detection/main/train.csv')

os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(TRAIN_PATH):
    print(f'Downloading train.csv from {TRAIN_URL}')
    print('This is ~94 MB, takes ~10-30 seconds depending on connection...')
    urllib.request.urlretrieve(TRAIN_URL, TRAIN_PATH)
    size_mb = os.path.getsize(TRAIN_PATH) / 1e6
    print(f'Download complete: {size_mb:.1f} MB')
else:
    print('train.csv already present, skipping download.')

# Load the CSV
df_raw = pd.read_csv(TRAIN_PATH)

# Sanity-check the schema and shape — fail loudly if the mirror has changed
expected_cols = ['id', 'title', 'author', 'text', 'label']
assert list(df_raw.columns) == expected_cols, (
    f'Schema mismatch! Expected {expected_cols}, got {list(df_raw.columns)}')
assert df_raw.shape[0] == 20800, (
    f'Row count mismatch! Expected 20800, got {df_raw.shape[0]}')

print(f'\nTotal articles loaded: {len(df_raw):,}')
print(f'Columns: {list(df_raw.columns)}')
print(f'Label distribution (raw, BEFORE flip): {df_raw["label"].value_counts().to_dict()}')
print(f'  -> 1 = fake/unreliable, 0 = real/reliable in this dataset')
print(f'\nFirst few rows:')
df_raw.head(3)

In [ ]:
# Inspect a sample article (label==0 means reliable in this dataset; we'll flip later)
sample_real = df_raw[df_raw['label'] == 0].iloc[0]
print('=== RELIABLE / REAL NEWS SAMPLE ===')
print(f"Title:  {sample_real['title']}")
print(f"Author: {sample_real['author']}")
print(f"Text (first 400 chars):\n{str(sample_real['text'])[:400]}")

In [ ]:
sample_fake = df_raw[df_raw['label'] == 1].iloc[0]
print('=== UNRELIABLE / FAKE NEWS SAMPLE ===')
print(f"Title:  {sample_fake['title']}")
print(f"Author: {sample_fake['author']}")
print(f"Text (first 400 chars):\n{str(sample_fake['text'])[:400]}")

## 3. Cleaning Up the Raw Dataframe

Three steps before modeling:

1. **Drop rows with missing `text`.** A small number of rows have NaN text (the article body is what we'll classify on; without it, the row is useless).
2. **Flip the labels.** The dataset uses `1 = fake, 0 = real`. We flip to `1 = real, 0 = fake` so the positive class is real news — consistent with the Bisaillon convention and with how we'd usually frame a classification problem ("can we identify real news?").
3. **Shuffle.** Random shuffle with a fixed seed so the train/test split downstream is well-mixed.

In [ ]:
# Drop rows missing the text body
n_before = len(df_raw)
df = df_raw.dropna(subset=['text']).reset_index(drop=True)
print(f'Dropped {n_before - len(df):,} rows with missing text. Remaining: {len(df):,}')

# Fill any remaining NaNs in title/author with empty strings (so concatenation works)
df['title'] = df['title'].fillna('')
df['author'] = df['author'].fillna('')

# Flip labels: original (1=fake, 0=real) -> (1=real, 0=fake)
df['label'] = 1 - df['label']

# Shuffle with fixed seed
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'\nClass distribution AFTER label flip (should match expected balance):')
print(df['label'].value_counts().rename({0: 'Fake', 1: 'Real'}))
print(f'\nClass balance: {df["label"].mean():.3f} real / {1 - df["label"].mean():.3f} fake')

## 4. Exploratory Data Analysis

### 4.1 Class Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
counts = df['label'].value_counts().rename({0: 'Fake', 1: 'Real'})
ax.bar(counts.index, counts.values, color=['#d62728', '#2ca02c'], alpha=0.8)
ax.set_ylabel('Number of articles')
ax.set_title('Class Distribution: Real vs. Fake News')
for i, v in enumerate(counts.values):
    ax.text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

### 4.2 Author Distribution

This dataset includes an `author` field — worth a quick look. If real and fake articles come from completely disjoint authors, the model could plausibly classify based on author identity rather than content. Let's check.

In [ ]:
# Top authors by class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, label, title, color in [(axes[0], 1, 'Real News', '#2ca02c'), 
                                  (axes[1], 0, 'Fake News', '#d62728')]:
    sub = df[df['label'] == label]
    # Drop empty author and count
    author_counts = sub[sub['author'] != '']['author'].value_counts().head(10)
    ax.barh(author_counts.index, author_counts.values, color=color, alpha=0.8)
    ax.set_title(f'{title} — Top 10 Authors')
    ax.set_xlabel('Number of articles')
    ax.invert_yaxis()
    ax.tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.show()

# Compute author overlap between classes
real_authors = set(df[(df['label']==1) & (df['author']!='')]['author'].unique())
fake_authors = set(df[(df['label']==0) & (df['author']!='')]['author'].unique())
overlap = real_authors & fake_authors
print(f'Unique real-news authors: {len(real_authors):,}')
print(f'Unique fake-news authors: {len(fake_authors):,}')
print(f'Authors appearing in BOTH classes: {len(overlap)}')
print(f'\nAuthor-overlap is small, which is normal — but to avoid the model learning author-specific style,')
print(f'we will only use the title and body text as features (NOT the author column).')

### 4.3 Article Length Distribution

We compare article lengths (in words) between classes. Differences in length distribution can be a discriminative signal — fake news is often either very short (clickbait) or padded.

In [ ]:
df['word_count'] = df['text'].apply(lambda x: len(str(x).split()))

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df[df['label'] == 1]['word_count'], bins=80, alpha=0.5, label='Real', color='#2ca02c', range=(0, 2000))
ax.hist(df[df['label'] == 0]['word_count'], bins=80, alpha=0.5, label='Fake', color='#d62728', range=(0, 2000))
ax.set_xlabel('Word count')
ax.set_ylabel('Number of articles')
ax.set_title('Article Length Distribution by Class (truncated at 2000 words)')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Real news word count -- median: {df[df['label']==1]['word_count'].median():.0f}, mean: {df[df['label']==1]['word_count'].mean():.0f}")
print(f"Fake news word count -- median: {df[df['label']==0]['word_count'].median():.0f}, mean: {df[df['label']==0]['word_count'].mean():.0f}")

### 4.4 Most Frequent Words Per Class (Before Cleaning)

A quick look at raw word frequency. This will be dominated by stopwords initially — that's expected and motivates the cleaning step in the next section.

In [ ]:
def top_words(texts, n=15):
    all_words = ' '.join(texts.astype(str)).lower().split()
    return Counter(all_words).most_common(n)

real_top = top_words(df[df['label'] == 1]['text'])
fake_top = top_words(df[df['label'] == 0]['text'])

comparison = pd.DataFrame({
    'Real - word': [w for w, _ in real_top],
    'Real - count': [c for _, c in real_top],
    'Fake - word': [w for w, _ in fake_top],
    'Fake - count': [c for _, c in fake_top],
})

print('Top 15 most frequent words (raw, before cleaning):')
print(comparison.to_string(index=False))

## 5. Text Preprocessing

We apply a four-step cleaning pipeline:

1. **Lowercase** all text
2. **Remove URLs, mentions, special characters, digits** — keep only letters and spaces
3. **Remove English stopwords** (the, of, and, ...) using NLTK's English stopword list
4. **Apply Porter stemming** to collapse morphological variants (running → run, votes → vote)

We concatenate the **title** and **text** fields as the input. We deliberately exclude the `author` column — even though it would likely improve accuracy, it would do so by learning author identity rather than content quality, which is not what we want.

In [ ]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def clean_text(text):
    """Full preprocessing pipeline applied to a single string."""
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)        # URLs
    text = re.sub(r'@\w+', ' ', text)                      # @mentions
    text = re.sub(r'\[.*?\]', ' ', text)                  # bracketed text e.g. [VIDEO]
    text = re.sub(r'<.*?>', ' ', text)                     # HTML tags
    text = re.sub(r'[^a-z\s]', ' ', text)                 # keep only letters
    text = re.sub(r'\s+', ' ', text).strip()              # collapse whitespace
    
    # Tokenize, drop stopwords, stem
    tokens = text.split()
    tokens = [stemmer.stem(t) for t in tokens 
              if t not in stop_words and len(t) > 1]
    return ' '.join(tokens)

# Test on one article first
print('--- Before cleaning ---')
print(str(df.iloc[0]['text'])[:300])
print('\n--- After cleaning ---')
print(clean_text(df.iloc[0]['text'])[:300])

In [ ]:
# Combine title + text (NOT author), then clean
# Cleaning ~21k articles takes ~1-2 minutes
print('Cleaning all articles... (this takes a couple of minutes)')
df['combined'] = df['title'].fillna('') + ' ' + df['text'].fillna('')
df['cleaned'] = df['combined'].apply(clean_text)

# Drop empty rows (very rare, but possible if an article was all stopwords/digits)
n_before_empty = len(df)
df = df[df['cleaned'].str.len() > 0].reset_index(drop=True)
print(f'Dropped {n_before_empty - len(df):,} rows with empty cleaned text. Remaining: {len(df):,}')

# FIX: Deduplicate on cleaned text BEFORE train/test split
# Earlier diagnostic detected 44 articles appearing in both train and test sets.
# Removing duplicates before splitting eliminates that leak entirely.
n_before_dedup = len(df)
df = df.drop_duplicates(subset=['cleaned']).reset_index(drop=True)
n_dropped = n_before_dedup - len(df)
print(f'Dropped {n_dropped:,} duplicate cleaned-text rows. Remaining: {len(df):,}')

if n_dropped > 0:
    print(f'  -> This prevents train/test leakage (where the same cleaned article appeared in both splits).')

In [ ]:
# Sanity check: top words per class AFTER cleaning
real_top = top_words(df[df['label'] == 1]['cleaned'])
fake_top = top_words(df[df['label'] == 0]['cleaned'])

comparison = pd.DataFrame({
    'Real - word': [w for w, _ in real_top],
    'Real - count': [c for _, c in real_top],
    'Fake - word': [w for w, _ in fake_top],
    'Fake - count': [c for _, c in fake_top],
})

print('Top 15 words AFTER cleaning (stems shown):')
print(comparison.to_string(index=False))

## 6. TF-IDF Vectorization

We use `TfidfVectorizer` with the following choices:

- **`max_features=20000`** — keep the 20,000 most frequent terms. This caps memory and reduces sparsity without sacrificing meaningful vocabulary on a 45k-document corpus.
- **`ngram_range=(1, 2)`** — include both unigrams and bigrams. Bigrams help capture phrases like `donald trump` or `breaking news` that carry meaning beyond their constituent words.
- **`min_df=5`** — drop terms appearing in fewer than 5 documents (typos, rare proper nouns).
- **`max_df=0.7`** — drop terms appearing in more than 70% of documents (corpus-level stopwords that survived our pipeline).
- **`sublinear_tf=True`** — apply log-scaling to term frequency, which is empirically beneficial for long documents.

## 7. Train/Test Split and Vectorization

**Important:** we fit the vectorizer **only on training data** to avoid test-set leakage through the IDF statistics. The fitted vectorizer is then applied to both train and test.

In [ ]:
# 80/20 stratified split
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['cleaned'].values,
    df['label'].values,
    test_size=0.2,
    stratify=df['label'].values,
    random_state=42,
)

print(f'Training set: {len(X_train_text):,} articles')
print(f'Test set:     {len(X_test_text):,} articles')
print(f'Train class balance: {y_train.mean():.3f}')
print(f'Test class balance:  {y_test.mean():.3f}')

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.7,
    sublinear_tf=True,
)

X_train_tfidf = vectorizer.fit_transform(X_train_text)
X_test_tfidf = vectorizer.transform(X_test_text)

print(f'TF-IDF training matrix shape: {X_train_tfidf.shape}')
print(f'TF-IDF test matrix shape:     {X_test_tfidf.shape}')
print(f'Density: {X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1]) * 100:.3f}%')

## 8. (Optional) Persisting Cleaned Data

Since the entire pipeline runs in one notebook, we don't strictly need to save artifacts to disk. However, saving the cleaned dataframe lets you skip the (~2 min) cleaning step if you restart the kernel mid-notebook.

In [ ]:
OUT_DIR = '/kaggle/working/'

# Save cleaned dataframe so we can skip cleaning on a kernel restart
# Note: this dataset has 'author' instead of 'subject' (vs. the Bisaillon dataset)
df[['title', 'author', 'text', 'cleaned', 'label']].to_csv(
    os.path.join(OUT_DIR, 'cleaned_data.csv'), index=False
)
print('Saved cleaned_data.csv to /kaggle/working/')

## 9. Stage 1 Complete — Ready for Modeling

**What we built in Stage 1:**

1. Downloaded the William Lifferth `fake-news` dataset (~21k articles, balanced).
2. Performed EDA on class distribution, author distribution, article length, and word frequency.
3. Cleaned the text: lowercased, stripped URLs/mentions/non-alphabetic characters, removed English stopwords, applied Porter stemming.
4. Performed an 80/20 stratified train/test split.
5. Fitted a TF-IDF vectorizer (unigrams + bigrams, 20k features) **on training data only**.

**Variables now available for the modeling stages:**

| Variable | Type | Description |
|---|---|---|
| `X_train_tfidf`, `X_test_tfidf` | sparse matrices | TF-IDF features (used for classical models) |
| `X_train_text`, `X_test_text` | numpy arrays of strings | Cleaned text (used by LSTM tokenizer) |
| `y_train`, `y_test` | numpy arrays | Binary labels (1 = real, 0 = fake) |
| `vectorizer` | fitted TfidfVectorizer | The TF-IDF vectorizer |

**Next: Stage 2 — Classical Models** (Multinomial Naive Bayes, Logistic Regression, Linear SVM, XGBoost) with cross-validated hyperparameter tuning, multi-metric evaluation, and a McNemar's test between the top two.

### 9b. Stage 1 Sanity Check — Verify Data is Set Up Correctly

Before running modeling stages, this cell confirms that all required variables exist with sensible shapes and values. If anything is wrong, this is where it'll show.

In [ ]:
# Sanity check: confirm everything Stage 2/3 needs is in scope and looks healthy
print('=' * 65)
print('  STAGE 1 SANITY CHECK')
print('=' * 65)
checks_passed = 0
total_checks = 8

# 1. df exists and has expected columns
try:
    assert 'cleaned' in df.columns and 'label' in df.columns
    print(f'  [PASS] df has cleaned and label columns ({len(df):,} rows)')
    checks_passed += 1
except (NameError, AssertionError) as e:
    print(f'  [FAIL] df missing or malformed: {e}')

# 2. label distribution
try:
    real_frac = df['label'].mean()
    assert 0.4 < real_frac < 0.6, f'Class imbalance: {real_frac:.3f}'
    print(f'  [PASS] label balance is {real_frac:.3f} real / {1-real_frac:.3f} fake (balanced)')
    checks_passed += 1
except (NameError, AssertionError) as e:
    print(f'  [FAIL] label distribution: {e}')

# 3. TF-IDF training matrix
try:
    assert X_train_tfidf.shape[0] > 10000, f'Training set too small: {X_train_tfidf.shape[0]}'
    assert X_train_tfidf.shape[1] == 20000, f'Wrong vocab size: {X_train_tfidf.shape[1]}'
    print(f'  [PASS] X_train_tfidf shape {X_train_tfidf.shape} (sparse, OK)')
    checks_passed += 1
except (NameError, AssertionError) as e:
    print(f'  [FAIL] X_train_tfidf: {e}')

# 4. TF-IDF test matrix
try:
    assert X_test_tfidf.shape[1] == X_train_tfidf.shape[1],         f'Train/test feature mismatch: {X_train_tfidf.shape[1]} vs {X_test_tfidf.shape[1]}'
    print(f'  [PASS] X_test_tfidf shape {X_test_tfidf.shape} (matches train vocab)')
    checks_passed += 1
except (NameError, AssertionError) as e:
    print(f'  [FAIL] X_test_tfidf: {e}')

# 5. y_train
try:
    assert len(y_train) == X_train_tfidf.shape[0]
    assert set(y_train).issubset({0, 1})
    print(f'  [PASS] y_train: {len(y_train)} labels, values in {{0,1}}')
    checks_passed += 1
except (NameError, AssertionError) as e:
    print(f'  [FAIL] y_train: {e}')

# 6. y_test
try:
    assert len(y_test) == X_test_tfidf.shape[0]
    assert set(y_test).issubset({0, 1})
    print(f'  [PASS] y_test: {len(y_test)} labels, values in {{0,1}}')
    checks_passed += 1
except (NameError, AssertionError) as e:
    print(f'  [FAIL] y_test: {e}')

# 7. X_train_text and X_test_text exist (for the LSTM)
try:
    assert len(X_train_text) == X_train_tfidf.shape[0]
    assert len(X_test_text) == X_test_tfidf.shape[0]
    print(f'  [PASS] X_train_text and X_test_text exist (needed for LSTM)')
    checks_passed += 1
except (NameError, AssertionError) as e:
    print(f'  [FAIL] X_train_text/X_test_text: {e}')

# 8. No data leakage between train and test (basic check)
try:
    train_set = set(X_train_text.tolist())
    test_set = set(X_test_text.tolist())
    overlap = train_set & test_set
    if len(overlap) == 0:
        print(f'  [PASS] No exact-duplicate articles between train and test')
        checks_passed += 1
    else:
        # Some duplicate articles in source data are possible — flag but don't fail
        print(f'  [WARN] {len(overlap)} articles appear in both train and test')
        print(f'         (may be benign duplicates in source data; investigate if results seem too good)')
        checks_passed += 1
except (NameError, AssertionError) as e:
    print(f'  [FAIL] leakage check: {e}')

print('=' * 65)
print(f'  {checks_passed}/{total_checks} checks passed')
if checks_passed == total_checks:
    print('  All systems go for Stage 2!')
else:
    print('  STOP — fix the failures above before proceeding to Stage 2.')
print('=' * 65)

# STAGE 2 — Classical Machine Learning Models

This stage trains and evaluates four classical models on the TF-IDF features prepared in Stage 1:

1. **Multinomial Naive Bayes** — fast probabilistic baseline, strong on text
2. **Logistic Regression** — strong discriminative linear baseline
3. **Linear Support Vector Machine** — max-margin alternative
4. **XGBoost** — gradient-boosted trees, included for comparison

Each model gets:
- Cross-validated hyperparameter tuning (grid or random search, **on training data only**)
- Multi-metric evaluation on the **held-out test set**: Accuracy, Precision, Recall, F1, MCC
- Confusion matrix
- Final selection via comparison + McNemar's test

---

## 🧭 How to Read the Output (CV vs. TEST — important!)

Every model produces TWO categories of numbers, and they are NOT the same thing:

| Output | Computed on | Used for | Bias |
|---|---|---|---|
| **`Best CV F1: X`** (printed during tuning) | 5-fold cross-validation **on training data** | Picking the best hyperparameters | Optimistic — was used to select the model |
| **`=== Model Name === Accuracy: X, F1: X` block** | The **held-out test set** | Final reported performance | Unbiased — model never saw this data |

**The numbers in the `=== Model Name ===` block are the headline test results** — those are what go in the paper.

The `Best CV F1` numbers are diagnostic: they tell us how well the model performs in cross-validation during tuning. **A large gap between CV F1 and test F1 indicates overfitting to the validation folds.** A gap of ~0–2 points is normal and healthy.

To make this distinction visually obvious, every model section will print a clearly-labeled comparison table immediately after evaluation.

## 10. Imports for the Modeling Stage

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, confusion_matrix,
                             classification_report)
from statsmodels.stats.contingency_tables import mcnemar
from scipy import stats

# Container for results — we'll aggregate all models' performance here at the end
results = {}

print('Modeling imports complete.')

## 11. Evaluation Helper Function

We define one function that takes a fitted model and returns all five metrics + the confusion matrix. Using a single helper ensures every model is evaluated identically.

The helper now also accepts a `cv_score` parameter — the best CV F1 from hyperparameter tuning — so that every model's output explicitly compares CV vs. Test F1 with a clear interpretation. This is critical because:

- **CV F1** measures performance on cross-validation folds *during* hyperparameter selection (it has selection bias — it was used to choose the model)
- **Test F1** measures performance on the held-out test set (no selection bias — clean estimate of real-world performance)
- **The gap (CV − Test)** indicates whether the model overfit to the CV folds during tuning. A gap of < 0.02 is healthy; > 0.05 suggests problematic overfitting.

In [ ]:
def evaluate_model(name, model, X_test, y_test, results_dict, cv_score=None):
    """Evaluate a fitted model on the test set, store results, and print a summary.
    
    Parameters
    ----------
    name : str
        Model name to use for display and storage
    model : fitted estimator
        Anything with .predict(X)
    X_test, y_test : test data
    results_dict : dict
        The global `results` dict to which we add `name` -> metrics
    cv_score : float, optional
        Best 5-fold CV F1 score from hyperparameter tuning. If provided,
        we'll print it alongside the test F1 to make CV-vs-test comparison
        explicit.
    """
    y_pred = model.predict(X_test)
    
    metrics = {
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall':    recall_score(y_test, y_pred),
        'f1':        f1_score(y_test, y_pred),
        'mcc':       matthews_corrcoef(y_test, y_pred),
        'cm':        confusion_matrix(y_test, y_pred),
        'y_pred':    y_pred,
        'cv_f1':     cv_score,
    }
    results_dict[name] = metrics
    
    print(f'\n{"=" * 60}')
    print(f'  {name.upper()} — TEST SET PERFORMANCE')
    print(f'{"=" * 60}')
    print(f'  These metrics are on the HELD-OUT TEST SET ({len(y_test):,} examples)')
    print(f'  The model was tuned on cross-validation folds of the')
    print(f'  TRAINING SET ONLY — it has never seen any test example.')
    print(f'{"-" * 60}')
    print(f'  Accuracy:  {metrics["accuracy"]:.4f}')
    print(f'  Precision: {metrics["precision"]:.4f}')
    print(f'  Recall:    {metrics["recall"]:.4f}')
    print(f'  F1:        {metrics["f1"]:.4f}')
    print(f'  MCC:       {metrics["mcc"]:.4f}')
    print(f'{"-" * 60}')
    print(f'  Confusion matrix on TEST SET (rows=true, cols=pred):')
    print(f'                Pred Fake  Pred Real')
    print(f'    True Fake:   {metrics["cm"][0,0]:6d}     {metrics["cm"][0,1]:6d}')
    print(f'    True Real:   {metrics["cm"][1,0]:6d}     {metrics["cm"][1,1]:6d}')
    
    # CV vs Test comparison if we have the CV score
    if cv_score is not None:
        gap = cv_score - metrics['f1']
        gap_sign = '+' if gap >= 0 else ''
        print(f'{"-" * 60}')
        print(f'  CV vs TEST F1 comparison:')
        print(f'    Best CV F1 (during tuning):  {cv_score:.4f}')
        print(f'    Test F1 (held-out):          {metrics["f1"]:.4f}')
        print(f'    Gap (CV − Test):             {gap_sign}{gap:+.4f}')
        if abs(gap) < 0.01:
            print(f'    -> Tiny gap: model generalizes excellently.')
        elif abs(gap) < 0.03:
            print(f'    -> Small gap: healthy generalization.')
        elif gap > 0:
            print(f'    -> CV > Test: mild overfit to CV folds during tuning.')
        else:
            print(f'    -> Test > CV: test set happens to be slightly easier than CV folds.')
    print(f'{"=" * 60}')
    return metrics

## 12. Model 1 — Multinomial Naive Bayes

**Theoretical framing.** Multinomial NB models per-class token-count distributions and applies Bayes' rule. In log-space, it reduces to a linear classifier with closed-form parameter estimates (no iterative optimization needed). Its only hyperparameter is the Laplace smoothing constant `alpha`, which controls how strongly the model regularizes toward uniform token probabilities.

**Tuning.** We grid-search `alpha` over five orders of magnitude with 5-fold CV.

In [ ]:
# Grid search on alpha
nb_param_grid = {'alpha': [3, 1, 0.1, 0.01, 0.001, 0.0001]}
nb_grid = GridSearchCV(
    estimator=MultinomialNB(),
    param_grid=nb_param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    verbose=1,
)
nb_grid.fit(X_train_tfidf, y_train)

print(f'\nBest alpha: {nb_grid.best_params_["alpha"]}')
print(f'Best CV F1: {nb_grid.best_score_:.4f}')

# Plot CV scores across alpha
fig, ax = plt.subplots(figsize=(8, 4))
alphas = nb_param_grid['alpha']
cv_scores = nb_grid.cv_results_['mean_test_score']
ax.plot(range(len(alphas)), cv_scores, marker='o', linewidth=2)
ax.set_xticks(range(len(alphas)))
ax.set_xticklabels([str(a) for a in alphas])
ax.set_xlabel('alpha (Laplace smoothing)')
ax.set_ylabel('5-fold CV F1 score')
ax.set_title('Naive Bayes: CV F1 vs. alpha')
ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate best NB model on test set
nb_best = nb_grid.best_estimator_
evaluate_model('Naive Bayes', nb_best, X_test_tfidf, y_test, results,
               cv_score=nb_grid.best_score_)

## 13. Model 2 — Logistic Regression

**Theoretical framing.** Logistic regression fits a linear classifier by minimizing L2-regularized negative log-likelihood. The hyperparameter `C` is the inverse regularization strength (small C = strong regularization).

**Tuning.** We grid-search `C` over six values with 5-fold CV.

In [ ]:
# Grid search on C
lr_param_grid = {'C': [0.01, 0.1, 1, 5, 10, 100]}
lr_grid = GridSearchCV(
    estimator=LogisticRegression(max_iter=1000, solver='liblinear'),
    param_grid=lr_param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    verbose=1,
)
lr_grid.fit(X_train_tfidf, y_train)

print(f'\nBest C: {lr_grid.best_params_["C"]}')
print(f'Best CV F1: {lr_grid.best_score_:.4f}')

In [ ]:
# Evaluate best LR model on test set
lr_best = lr_grid.best_estimator_
evaluate_model('Logistic Regression', lr_best, X_test_tfidf, y_test, results,
               cv_score=lr_grid.best_score_)

## 14. Model 3 — Linear Support Vector Machine

**Theoretical framing.** Linear SVM finds the maximum-margin hyperplane separating the two classes, with `C` controlling tolerance for margin violations. We use `LinearSVC` (rather than `SVC(kernel='linear')`) because it's substantially faster on large sparse data — internally it uses a different optimization library (`liblinear` rather than `libsvm`) optimized for this case.

**Tuning.** Grid search on `C` with 5-fold CV.

In [ ]:
# Grid search on C for LinearSVC
svm_param_grid = {'C': [0.01, 0.1, 1, 5, 10]}
svm_grid = GridSearchCV(
    estimator=LinearSVC(max_iter=2000),
    param_grid=svm_param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    verbose=1,
)
svm_grid.fit(X_train_tfidf, y_train)

print(f'\nBest C: {svm_grid.best_params_["C"]}')
print(f'Best CV F1: {svm_grid.best_score_:.4f}')

In [ ]:
# Evaluate best SVM on test set
svm_best = svm_grid.best_estimator_
evaluate_model('Linear SVM', svm_best, X_test_tfidf, y_test, results,
               cv_score=svm_grid.best_score_)

## 15. Model 4 — XGBoost

**Theoretical framing.** XGBoost is a gradient-boosted tree ensemble. Trees are added sequentially, each correcting the residual error of the previous ensemble.

**Tuning.** XGBoost has many hyperparameters; we use `RandomizedSearchCV` to sample 15 configurations rather than running a full grid search. We tune `max_depth`, `n_estimators`, `learning_rate`, and `min_child_weight`.

**Expected result.** Based on prior experience with text classification (and your previous projects), XGBoost typically underperforms linear models on TF-IDF features. We include it primarily as a calibration point — to confirm that the linear models are doing as well as they should be.

In [ ]:
# Randomized search for XGBoost
# Note: we set n_jobs=2 (rather than -1) for the RandomizedSearchCV outer loop
# to avoid worker-killed warnings from memory pressure when running 4+ XGBoost
# models in parallel. The inner XGBClassifier still uses all CPU cores via its own n_jobs.
xgb_param_dist = {
    'max_depth':        [4, 6, 8, 10],
    'n_estimators':     [100, 200, 300],
    'learning_rate':    [0.05, 0.1, 0.2, 0.3],
    'min_child_weight': [1, 3, 5],
}

xgb_random = RandomizedSearchCV(
    estimator=XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        n_jobs=-1,
        random_state=42,
    ),
    param_distributions=xgb_param_dist,
    n_iter=15,
    scoring='f1',
    cv=3,           # 3-fold (rather than 5) to keep training time manageable
    n_jobs=2,       # was -1; reduced to 2 to avoid OOM-related worker deaths
    random_state=42,
    verbose=1,
)
xgb_random.fit(X_train_tfidf, y_train)

print(f'\nBest params: {xgb_random.best_params_}')
print(f'Best CV F1: {xgb_random.best_score_:.4f}')

In [ ]:
# Evaluate best XGBoost on test set
xgb_best = xgb_random.best_estimator_
evaluate_model('XGBoost', xgb_best, X_test_tfidf, y_test, results,
               cv_score=xgb_random.best_score_)

## 16. Model Comparison

We compare all four models side-by-side across all five metrics.

In [ ]:
# Build comparison DataFrame WITH explicit CV-vs-Test columns
metrics_df = pd.DataFrame({
    name: {
        'CV F1 (training)':   m.get('cv_f1', None),
        'Test Accuracy':      m['accuracy'],
        'Test Precision':     m['precision'],
        'Test Recall':        m['recall'],
        'Test F1':            m['f1'],
        'Test MCC':           m['mcc'],
        'CV-Test F1 gap':     (m.get('cv_f1') - m['f1']) if m.get('cv_f1') is not None else None,
    }
    for name, m in results.items()
}).T

print('=' * 75)
print('CLASSICAL MODELS — CV vs TEST PERFORMANCE COMPARISON')
print('=' * 75)
print('Note: CV F1 is computed on training-set folds during hyperparameter tuning.')
print('      Test F1 is computed on the held-out test set (never seen during tuning).')
print('      A small gap (< 0.02) indicates healthy generalization.')
print('=' * 75)
print(metrics_df.round(4))

In [ ]:
# Bar chart: CV F1 vs Test F1 vs Test Accuracy for all classical models
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

model_names = list(results.keys())
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

# LEFT: Test accuracy
ax = axes[0]
accs = [results[m]['accuracy'] for m in model_names]
bars = ax.bar(model_names, accs, color=colors, alpha=0.85)
ax.set_ylim(0.5, 1.05)
ax.set_ylabel('Test Accuracy')
ax.set_title('Test Accuracy by Model (held-out 4143 articles)')
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{acc:.4f}', ha='center', fontweight='bold', fontsize=10)
ax.grid(axis='y', alpha=0.3)

# RIGHT: CV F1 vs Test F1 — shows generalization gap
ax = axes[1]
x_positions = np.arange(len(model_names))
width = 0.35
cv_f1s   = [results[m].get('cv_f1', 0) for m in model_names]
test_f1s = [results[m]['f1'] for m in model_names]
b1 = ax.bar(x_positions - width/2, cv_f1s, width, color='#88B0D6', label='CV F1 (training folds)')
b2 = ax.bar(x_positions + width/2, test_f1s, width, color='#D67888', label='Test F1 (held-out)')
ax.set_xticks(x_positions)
ax.set_xticklabels(model_names, rotation=10, ha='right')
ax.set_ylim(0.5, 1.05)
ax.set_ylabel('F1 Score')
ax.set_title('CV vs Test F1 — generalization check')
ax.legend(loc='lower right')
for bars in [b1, b2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', fontsize=8)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/classical_models_cv_vs_test.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion matrices side-by-side for all four models
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, (name, m) in zip(axes.flat, results.items()):
    cm = m['cm']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Fake', 'Real'], yticklabels=['Fake', 'Real'],
                cbar=False)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(f'{name}  (acc={m["accuracy"]:.4f})')
plt.tight_layout()
plt.show()

## 17. McNemar's Test — Are the Top Two Models Statistically Different?

When two models perform similarly, raw accuracy comparison can mislead. **McNemar's test** asks whether the disagreement between two models is symmetric: if Model A is wrong where Model B is right at roughly the same rate as the reverse, the two are statistically equivalent. If one direction dominates, they're meaningfully different.

We run McNemar's test between the top two models by F1 score.

In [ ]:
# Identify the top two models by F1
sorted_models = sorted(results.items(), key=lambda x: x[1]['f1'], reverse=True)
top_two = sorted_models[:2]
name_a, name_b = top_two[0][0], top_two[1][0]
pred_a = top_two[0][1]['y_pred']
pred_b = top_two[1][1]['y_pred']

# Build a 2x2 contingency table:
#   Rows = Model A correctness, Cols = Model B correctness
correct_a = (pred_a == y_test)
correct_b = (pred_b == y_test)

n_both_right     = ((correct_a) & (correct_b)).sum()
n_a_right_b_wrong = ((correct_a) & (~correct_b)).sum()
n_a_wrong_b_right = ((~correct_a) & (correct_b)).sum()
n_both_wrong    = ((~correct_a) & (~correct_b)).sum()

contingency = np.array([[n_both_right, n_a_right_b_wrong],
                         [n_a_wrong_b_right, n_both_wrong]])

print(f'Comparing top two models: {name_a}  vs.  {name_b}')
print(f'\nContingency table:')
print(f'                       {name_b} right  {name_b} wrong')
print(f'  {name_a:<18} right  {contingency[0,0]:8d}      {contingency[0,1]:8d}')
print(f'  {name_a:<18} wrong  {contingency[1,0]:8d}      {contingency[1,1]:8d}')

# McNemar's test (using the exact binomial form, which is appropriate for any sample size)
mcn_result = mcnemar(contingency, exact=True)
print(f'\nMcNemar test statistic: {mcn_result.statistic}')
print(f'p-value: {mcn_result.pvalue:.6f}')

if mcn_result.pvalue < 0.05:
    print(f'\np < 0.05 -> reject H0. The two models make systematically different predictions.')
else:
    print(f'\np >= 0.05 -> fail to reject H0. The two models are statistically equivalent.')

## 18. Stage 2 Complete — Classical Models Done

**Summary of classical results:**

We tuned four models with cross-validated hyperparameter search and evaluated each on the held-out test set across five metrics. The accuracy and F1 results, the confusion matrices, and the McNemar's test give us a complete picture of which classical model performs best — and whether the differences between the top contenders are statistically meaningful.

**Coming next: Stage 3 — LSTM**

In Stage 3 we build a simple bidirectional LSTM in Keras, train it from scratch on the cleaned text (without TF-IDF — LSTMs need a token sequence, not a bag-of-words vector), and compare its test performance to the classical winners.

**Important note for Stage 3:** the LSTM uses `X_train_text` and `X_test_text` (the cleaned strings) rather than the TF-IDF matrices.

# STAGE 3 — Bidirectional LSTM

This stage trains a Bidirectional LSTM as the deep learning component of our comparison. Unlike the classical models in Stage 2, the LSTM operates on **integer-encoded token sequences** rather than TF-IDF vectors, so we need a different vectorization step.

**Pipeline:**
1. Tokenize the cleaned text into integer sequences using Keras's `Tokenizer`
2. Pad/truncate sequences to a fixed length
3. Build a small Bidirectional LSTM (embedding → BiLSTM → Dense)
4. Train with early stopping, evaluate using the same `evaluate_model` helper as Stage 2
5. Plot training/validation loss curves

**Architecture choice.** We use a small Bidirectional LSTM (one layer, 64 units) with a trainable embedding layer (no pre-trained word vectors). This keeps training fast (~5–10 minutes on Kaggle's CPU; under 2 minutes on GPU) while still demonstrating sequential modeling capacity beyond bag-of-words representations.

## 19. Imports for the LSTM Stage

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# Set random seeds for partial reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Suppress TF info logging (keeps output cleaner)
tf.get_logger().setLevel('ERROR')

print(f'TensorFlow version: {tf.__version__}')

# FIX: Explicit GPU configuration. The previous run crashed cuInit with error 303
# and silently fell back to CPU, making LSTM training ~10x slower. We now
# explicitly enumerate GPUs, set memory_growth=True, and report what we got.
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f'GPU(s) detected and configured: {[g.name for g in gpus]}')
        print('LSTM training will use GPU acceleration.')
    except RuntimeError as e:
        print(f'GPU config failed: {e}')
        print('LSTM will fall back to CPU (slow but works).')
else:
    print('No GPU detected. LSTM will run on CPU (much slower).')
    print('If this is unexpected, check that the Kaggle accelerator is set to GPU P100.')

## 20. Tokenization and Sequence Padding

Keras's `Tokenizer` builds a word-to-integer mapping from the training texts, then converts each text into a list of integers. We then pad/truncate to a fixed length so all sequences have the same shape (required for batched training).

**Hyperparameter choices:**
- `MAX_VOCAB_SIZE = 20000` — keep the 20k most frequent tokens (matches our TF-IDF vocab size for fair comparison)
- `MAX_LEN = 300` — truncate articles to 300 tokens. Most cleaned articles are well under this; very long articles get clipped to the first 300 tokens, which is sufficient for sentiment-style discrimination.

In [ ]:
MAX_VOCAB_SIZE = 20000
MAX_LEN = 500       # was 300; raised because median article length is 311 tokens
                    # (51% of articles previously got truncated; with 500 it drops to ~25%)
EMBED_DIM = 64

# Fit tokenizer on training text only (avoid test-set leakage)
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train_text)

# Convert texts to integer sequences
X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq  = tokenizer.texts_to_sequences(X_test_text)

# Pad sequences to fixed length
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_LEN, padding='post', truncating='post')

print(f'Vocabulary size: {min(len(tokenizer.word_index), MAX_VOCAB_SIZE):,}')
print(f'Training sequences shape: {X_train_pad.shape}')
print(f'Test sequences shape:     {X_test_pad.shape}')

# Sequence length stats
seq_lengths = [len(s) for s in X_train_seq]
print(f'\nTraining sequence length stats (before padding):')
print(f'  median: {np.median(seq_lengths):.0f}')
print(f'  mean:   {np.mean(seq_lengths):.0f}')
print(f'  95th %: {np.percentile(seq_lengths, 95):.0f}')
print(f'  max:    {max(seq_lengths)}')
trunc_frac = np.mean(np.array(seq_lengths) > MAX_LEN)
print(f'\nFraction of articles longer than MAX_LEN={MAX_LEN}: {trunc_frac:.1%}')
print(f'  -> these will be truncated to the first {MAX_LEN} tokens.')

## 21. Build and Train the Bidirectional LSTM

**Architecture:**
- `Embedding` (vocab_size × 64) — learnable word embeddings
- `Bidirectional(LSTM(64))` — reads sequence in both directions, captures context
- `Dropout(0.5)` — strong regularization to prevent overfitting on small dataset
- `Dense(32, relu)` — small hidden layer
- `Dense(1, sigmoid)` — binary classification head

**Training:**
- Optimizer: Adam with default learning rate (1e-3)
- Loss: binary cross-entropy
- Batch size: 64
- Max 10 epochs with early stopping (patience=2 on validation loss)
- 15% of training data held out as a validation set

In [ ]:
def build_lstm_model(vocab_size, embed_dim, max_len):
    """Bidirectional LSTM with stronger regularization.
    
    Changes from the previous version (which overfit hard, val_loss diverging from epoch 1):
    - Smaller LSTM (64 -> 48 units) reduces parameter count
    - recurrent_dropout=0.2 inside the LSTM cell (regularizes hidden-state transitions)
    - Higher dropout between layers (0.5 -> 0.6 after LSTM, 0.3 -> 0.4 after Dense)
    - Lower learning rate (1e-3 -> 5e-4) for more stable convergence
    """
    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=embed_dim, input_length=max_len),
        Bidirectional(LSTM(48, recurrent_dropout=0.2)),
        Dropout(0.6),
        Dense(32, activation='relu'),
        Dropout(0.4),
        Dense(1, activation='sigmoid'),
    ])
    model.compile(
        optimizer=Adam(learning_rate=5e-4),
        loss='binary_crossentropy',
        metrics=['accuracy'],
    )
    return model

vocab_size = min(len(tokenizer.word_index) + 1, MAX_VOCAB_SIZE)
lstm_model = build_lstm_model(vocab_size, EMBED_DIM, MAX_LEN)
lstm_model.summary()

In [ ]:
# Train with stronger early-stopping policy and learning-rate reduction
# Previously: patience=2, no LR scheduling -> training stopped at epoch 3 with restored
# epoch 1 weights, meaning the model effectively trained for 1 useful epoch.
# Now: patience=3 (let it explore further), and ReduceLROnPlateau will halve LR
# if val_loss plateaus, giving the model a chance to fine-tune.
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,                  # was 2
    restore_best_weights=True,
    verbose=1,
)
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,                  # halve the LR when val_loss stalls
    patience=1,                  # react quickly
    min_lr=1e-5,
    verbose=1,
)

history = lstm_model.fit(
    X_train_pad, y_train,
    validation_split=0.15,
    epochs=10,
    batch_size=64,
    callbacks=[early_stop, reduce_lr],
    verbose=1,
)

print(f'\nTraining stopped after {len(history.history["loss"])} epochs.')

# Compute internal-validation F1 explicitly (Keras only tracks accuracy, not F1).
# This serves as the LSTM's "CV-equivalent" score in the final comparison table.
val_split = 0.15
n_val = int(len(X_train_pad) * val_split)
X_val_internal = X_train_pad[-n_val:]
y_val_internal = y_train[-n_val:]
val_pred = (lstm_model.predict(X_val_internal, verbose=0).ravel() >= 0.5).astype(int)
lstm_val_f1 = f1_score(y_val_internal, val_pred)
print(f'LSTM internal-validation F1: {lstm_val_f1:.4f}')

## 22. Training Curves

We plot training and validation loss/accuracy across epochs. A widening gap between training and validation curves indicates overfitting; ideally both curves track each other closely.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss curves
ax = axes[0]
ax.plot(history.history['loss'], label='Train loss', marker='o')
ax.plot(history.history['val_loss'], label='Val loss', marker='s')
ax.set_xlabel('Epoch')
ax.set_ylabel('Binary cross-entropy loss')
ax.set_title('LSTM Training Loss')
ax.legend()
ax.grid(True)

# Accuracy curves
ax = axes[1]
ax.plot(history.history['accuracy'], label='Train acc', marker='o')
ax.plot(history.history['val_accuracy'], label='Val acc', marker='s')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title('LSTM Training Accuracy')
ax.legend()
ax.grid(True)

plt.tight_layout()
plt.savefig('/kaggle/working/lstm_training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

## 23. Evaluate the LSTM on the Test Set

We use the same `evaluate_model` helper from Stage 2 so the LSTM result is directly comparable to the classical models.

**Note:** the LSTM outputs probabilities (sigmoid activation), so we wrap it in a thin class that adds a `.predict()` method returning hard 0/1 labels — matching the sklearn API the helper expects.

In [ ]:
class LSTMWrapper:
    """Thin wrapper to match sklearn's predict() API: returns hard 0/1 labels."""
    def __init__(self, keras_model):
        self.keras_model = keras_model
    def predict(self, X):
        probs = self.keras_model.predict(X, verbose=0).ravel()
        return (probs >= 0.5).astype(int)

lstm_wrapped = LSTMWrapper(lstm_model)
# Pass the internal-validation F1 as the LSTM's "CV-equivalent" score
evaluate_model('LSTM', lstm_wrapped, X_test_pad, y_test, results,
               cv_score=lstm_val_f1)

# STAGE 4 — Final Comparison and Summary

All five models trained and evaluated. We now produce the final comparison table, a unified bar chart, and an error-analysis step that surfaces example articles where each model failed.

## 24. Final Results Table

In [ ]:
# Build final comparison DataFrame including the LSTM, with explicit CV vs Test
final_metrics_df = pd.DataFrame({
    name: {
        'CV/Val F1':       m.get('cv_f1'),
        'Test Accuracy':   m['accuracy'],
        'Test Precision':  m['precision'],
        'Test Recall':     m['recall'],
        'Test F1':         m['f1'],
        'Test MCC':        m['mcc'],
        'CV-Test F1 gap':  (m.get('cv_f1') - m['f1']) if m.get('cv_f1') is not None else None,
    }
    for name, m in results.items()
}).T.round(4)

print('=' * 80)
print('FINAL TEST SET PERFORMANCE — ALL 5 MODELS')
print('=' * 80)
print('Note: "CV/Val F1" is the validation F1 score from hyperparameter tuning.')
print('      For classical models: 5-fold cross-validation on the training set.')
print('      For LSTM: 15% internal validation split during training.')
print('      The TEST columns are the headline numbers (held-out 4143 articles).')
print('=' * 80)
print(final_metrics_df)
print('=' * 80)

# Identify the winner by Test F1 (the headline metric)
best_model_name = final_metrics_df['Test F1'].idxmax()
print(f'\nBest model by Test F1: {best_model_name}  (Test F1 = {final_metrics_df.loc[best_model_name, "Test F1"]:.4f})')

# Save to CSV for the paper
final_metrics_df.to_csv('/kaggle/working/final_results_table.csv')
print('\nSaved to /kaggle/working/final_results_table.csv')

## 25. Unified Comparison Chart

In [ ]:
# Side-by-side: Test Accuracy and CV-vs-Test F1 for all 5 models
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

model_names = list(results.keys())
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

# LEFT: Test accuracy by model
ax = axes[0]
accs = [results[m]['accuracy'] for m in model_names]
bars = ax.bar(model_names, accs, color=colors, alpha=0.85)
ax.set_ylim(0.5, 1.05)
ax.set_ylabel('Test Accuracy')
ax.set_title('Test Accuracy — All 5 Models (held-out test set)')
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{acc:.4f}', ha='center', fontweight='bold', fontsize=10)
ax.tick_params(axis='x', rotation=15)
ax.grid(axis='y', alpha=0.3)

# RIGHT: CV/Val F1 vs Test F1 (shows generalization gap)
ax = axes[1]
x_positions = np.arange(len(model_names))
width = 0.35
cv_f1s   = [results[m].get('cv_f1', 0) or 0 for m in model_names]
test_f1s = [results[m]['f1'] for m in model_names]
b1 = ax.bar(x_positions - width/2, cv_f1s, width, color='#88B0D6', label='CV/Val F1')
b2 = ax.bar(x_positions + width/2, test_f1s, width, color='#D67888', label='Test F1')
ax.set_xticks(x_positions)
ax.set_xticklabels(model_names, rotation=10, ha='right')
ax.set_ylim(0.5, 1.05)
ax.set_ylabel('F1 Score')
ax.set_title('CV/Val F1 vs Test F1 — generalization check')
ax.legend(loc='lower right')
for bars in [b1, b2]:
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.01,
                    f'{h:.3f}', ha='center', fontsize=8)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/final_comparison_chart.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion matrices for all 5 models in one grid
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, (name, m) in zip(axes.flat, results.items()):
    cm = m['cm']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Fake', 'Real'], yticklabels=['Fake', 'Real'],
                cbar=False)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(f'{name}  (acc={m["accuracy"]:.4f})')

# Hide the unused 6th subplot
axes.flat[-1].set_visible(False)

plt.tight_layout()
plt.savefig('/kaggle/working/all_confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()

## 26. Error Analysis — Example Misclassifications

A useful qualitative complement to the quantitative metrics: for the best model, look at a few articles it got wrong. Are the errors clustered around any particular pattern (e.g., short articles, particular subjects, ambiguous phrasing)?

In [ ]:
# Get the best model's predictions
best_pred = results[best_model_name]['y_pred']

# Find indices of test-set errors
error_mask = best_pred != y_test
error_indices = np.where(error_mask)[0]

print(f'Total errors by {best_model_name}: {error_mask.sum()} / {len(y_test)} ({error_mask.mean():.2%})')

# Sample a few errors of each type
fp_indices = np.where(error_mask & (y_test == 0))[0]   # predicted real, actually fake
fn_indices = np.where(error_mask & (y_test == 1))[0]   # predicted fake, actually real

print(f'  False positives (predicted Real, actually Fake): {len(fp_indices)}')
print(f'  False negatives (predicted Fake, actually Real): {len(fn_indices)}')

# Show 3 examples of each
def show_errors(name, indices, n=3):
    print(f'\n{"=" * 70}')
    print(f'{name} — {n} examples')
    print("=" * 70)
    for idx in np.random.RandomState(42).choice(indices, size=min(n, len(indices)), replace=False):
        # Pull the original (uncleaned) article from df via positional index in test set
        # We need to look up the original index — easiest is to print the cleaned text
        print(f'\n[Test idx {idx}]  cleaned text (first 250 chars):')
        print(f'  {X_test_text[idx][:250]}...')

if len(fp_indices) > 0:
    show_errors('FALSE POSITIVES (Real predicted, Fake actual)', fp_indices, n=3)
if len(fn_indices) > 0:
    show_errors('FALSE NEGATIVES (Fake predicted, Real actual)', fn_indices, n=3)

## 27. Export Results for Reporting

All artifacts needed for the paper write-up are saved to `/kaggle/working/` as files you can download:

- `final_results_table.csv` — five-model performance table
- `final_comparison_chart.png` — accuracy and F1 bar chart
- `all_confusion_matrices.png` — confusion matrix grid
- `lstm_training_curves.png` — LSTM loss/accuracy curves
- `all_model_predictions.csv` — every test-set example with predictions from each model
- `mcnemar_test_result.txt` — McNemar's test output between the top two classical models
- `experiment_summary.txt` — one-page text summary of the entire experiment

To download from Kaggle: in the right sidebar, expand the **Output** section. Each file has a download button (or click "Download All" to grab them as a zip).

In [ ]:
import os, json

OUT_DIR = '/kaggle/working/'

# 1. Dump per-example predictions for all 5 models (useful for paper-stage error analysis)
predictions_df = pd.DataFrame({
    'cleaned_text': X_test_text,
    'true_label':   y_test,
})
for name, m in results.items():
    predictions_df[f'pred_{name.replace(" ", "_")}'] = m['y_pred']
predictions_df.to_csv(os.path.join(OUT_DIR, 'all_model_predictions.csv'), index=False)

# 2. McNemar test result (between top two by F1)
sorted_models = sorted(results.items(), key=lambda x: x[1]['f1'], reverse=True)
top_two = sorted_models[:2]
name_a, name_b = top_two[0][0], top_two[1][0]
pred_a = top_two[0][1]['y_pred']
pred_b = top_two[1][1]['y_pred']

correct_a = (pred_a == y_test)
correct_b = (pred_b == y_test)
contingency_top = np.array([
    [((correct_a)  & (correct_b)).sum(),  ((correct_a)  & (~correct_b)).sum()],
    [((~correct_a) & (correct_b)).sum(), ((~correct_a) & (~correct_b)).sum()],
])
mcn = mcnemar(contingency_top, exact=True)

with open(os.path.join(OUT_DIR, 'mcnemar_test_result.txt'), 'w') as f:
    f.write(f'McNemar test: {name_a} vs. {name_b}\n')
    f.write(f'Contingency table:\n')
    f.write(f'                     {name_b} right  {name_b} wrong\n')
    f.write(f'  {name_a:<18} right  {contingency_top[0,0]:8d}      {contingency_top[0,1]:8d}\n')
    f.write(f'  {name_a:<18} wrong  {contingency_top[1,0]:8d}      {contingency_top[1,1]:8d}\n')
    f.write(f'\nStatistic: {mcn.statistic}\n')
    f.write(f'p-value:   {mcn.pvalue:.6f}\n')
    f.write(f'\nInterpretation: ')
    if mcn.pvalue < 0.05:
        f.write(f'p < 0.05 -> reject H0. The two models make systematically different predictions.\n')
    else:
        f.write(f'p >= 0.05 -> fail to reject H0. The two models are statistically equivalent.\n')

# 3. One-page experiment summary (everything we'd need to write the paper)
summary_lines = []
summary_lines.append('FAKE NEWS DETECTION — EXPERIMENT SUMMARY')
summary_lines.append('=' * 70)
summary_lines.append('')
summary_lines.append('Dataset:')
summary_lines.append(f'  William Lifferth Fake News Dataset (Kaggle competition: fake-news)')
summary_lines.append(f'  Total articles: {len(df):,}')
summary_lines.append(f'  Real (label=1): {(df["label"]==1).sum():,}')
summary_lines.append(f'  Fake (label=0): {(df["label"]==0).sum():,}')
summary_lines.append('')
summary_lines.append('Train/test split: 80/20 stratified, random_state=42')
summary_lines.append(f'  Training set: {len(X_train_text):,}')
summary_lines.append(f'  Test set:     {len(X_test_text):,}')
summary_lines.append('')
summary_lines.append('TF-IDF features (classical models):')
summary_lines.append(f'  max_features=20000, ngram_range=(1,2), min_df=5, max_df=0.7, sublinear_tf=True')
summary_lines.append('')
summary_lines.append('LSTM features:')
summary_lines.append(f'  Vocab size: {min(len(tokenizer.word_index), MAX_VOCAB_SIZE):,}')
summary_lines.append(f'  Sequence length: {MAX_LEN}')
summary_lines.append(f'  Embedding dim: {EMBED_DIM}')
summary_lines.append('')
summary_lines.append('=' * 70)
summary_lines.append('FINAL RESULTS — ALL 5 MODELS')
summary_lines.append('(CV/Val F1 is from hyperparameter tuning; Test columns are held-out)')
summary_lines.append('=' * 70)
summary_lines.append(final_metrics_df.to_string())
summary_lines.append('')
summary_lines.append(f'Best model by F1: {best_model_name}  (F1 = {final_metrics_df.loc[best_model_name, "F1"]:.4f})')
summary_lines.append('')
summary_lines.append('=' * 70)
summary_lines.append('BEST HYPERPARAMETERS')
summary_lines.append('=' * 70)
summary_lines.append(f'Naive Bayes:         alpha = {nb_grid.best_params_["alpha"]}')
summary_lines.append(f'Logistic Regression: C = {lr_grid.best_params_["C"]}')
summary_lines.append(f'Linear SVM:          C = {svm_grid.best_params_["C"]}')
summary_lines.append(f'XGBoost:             {xgb_random.best_params_}')
summary_lines.append(f'LSTM:                Bidirectional(LSTM(64)) -> Dense(32) -> Dense(1)')
summary_lines.append(f'                     Trained for {len(history.history["loss"])} epochs (early-stopped)')
summary_lines.append('')
summary_lines.append('=' * 70)
summary_lines.append('McNemar TEST (top two classical models)')
summary_lines.append('=' * 70)
summary_lines.append(f'  {name_a} vs. {name_b}')
summary_lines.append(f'  p-value: {mcn.pvalue:.6f}')
summary_lines.append(f'  Decision at alpha=0.05: ' + ('reject H0 (different)' if mcn.pvalue < 0.05 else 'fail to reject (equivalent)'))

with open(os.path.join(OUT_DIR, 'experiment_summary.txt'), 'w') as f:
    f.write('\n'.join(summary_lines))

# Print the summary so it's visible in the notebook output
print('\n'.join(summary_lines))

# 4. List all files in /kaggle/working/ that should be downloaded
print('\n' + '=' * 70)
print('FILES TO DOWNLOAD FROM /kaggle/working/:')
print('=' * 70)
for fname in sorted(os.listdir(OUT_DIR)):
    fpath = os.path.join(OUT_DIR, fname)
    if os.path.isfile(fpath):
        size_kb = os.path.getsize(fpath) / 1024
        print(f'  {fname:40s} {size_kb:>10.1f} KB')

## 28. End of Notebook

**You now have everything needed for the paper.** Download the following files from `/kaggle/working/` (right sidebar of the Kaggle interface, click "Output"):

1. `final_results_table.csv` — main results table
2. `experiment_summary.txt` — full experiment summary
3. `mcnemar_test_result.txt` — McNemar test details
4. `final_comparison_chart.png` — accuracy/F1 bar chart
5. `all_confusion_matrices.png` — confusion matrix grid
6. `lstm_training_curves.png` — LSTM training curves
7. `all_model_predictions.csv` — per-example predictions (for any further error analysis)

**Once you've downloaded these and uploaded them back to me**, I'll write the paper based on your actual results.